In [ ]:
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from esdglider import defaults, gcp, paths

deployment_name = "amlr01-20191206"; mode = "delayed"

home = Path.home()
glider_paths = paths.get_path_glider(
    deployment_name = deployment_name, 
    mode = mode, 
    home_path = home,
)
gcp.gcs_mount_bucket_check(defaults.BUCKET_DATA_OUT, glider_paths["data_out_path"], ro=True)

### Raw timeseries, and profile summary

In [ ]:
ds_raw = xr.load_dataset(glider_paths["tsrawpath"])
display(ds_raw)

In [ ]:
prof_summ = pd.read_csv(glider_paths["profsummpath"])
prof_summ

### Engineering timeseries

In [ ]:
ds_eng = xr.load_dataset(glider_paths["tsengpath"])
# df_eng = ds_eng.to_pandas()
ds_eng

### Science timeseries

In [ ]:
ds_sci = xr.load_dataset(glider_paths["tsscipath"])
# df_sci = ds_sci.to_pandas()
ds_sci

In [ ]:
print(ds_sci.attrs["id"])
print(set(ds_sci["trajectory"].values))

In [ ]:
x = xr.load_dataset(glider_paths["gr5path"])
# df_sci = ds_sci.to_pandas()
x

### Profiles

In [ ]:
# List profile files
profile_files = sorted(
    Path(glider_paths["ngdacdir"]).glob("*.nc")
)
for profile_file in profile_files[:5]:
    print(profile_file.name)

# Load the first file, to check attributes
profile_file0 = profile_files[0]

prof0 = xr.load_dataset(
    profile_file0,
)

prof0

In [ ]:
prof0["trajectory"]

In [ ]:
# import netCDF4

# # Open the NetCDF file
# print(profile_file0)
# with netCDF4.Dataset(profile_file0, "r") as src:
#     # Generate the CDL text representation (attributes, dimensions, variables)
#     # Pass 'data=False' if you only want the metadata/attributes without data blocks
#     cdl_text = prof0.tocdl(data=False)

# cdl_text
# # # Write the CDL string to a text file
# # with open("output_file.cdl", "w") as f:
# #     f.write(cdl_text)

with open(home / f"{profile_file0.stem}.cdl", "w") as cdl_file:
    subprocess.run(["ncdump", "-h",  profile_file0], stdout=cdl_file, check=True)

### Issues

Examples of filtering datasets by variable value, or time slice

In [ ]:
x = ds_sci.copy(deep=True)

# Filter by variable value
x_mask = (x["latitude"]<32.47) & (x["longitude"]<-118) & (x["longitude"]>-118.2)
x_mask2 = (x["latitude"]<32.47) & (x["longitude"]<-118) & (x["longitude"]>-118.2)

x = x.where(x_mask, drop=True)

x

In [ ]:
# Filter by time slice
y = ds_sci.sel(time=slice('2026-04-13 01:42:00', '2026-04-13 14:30:00'))
y

Plot all science data points, overlaid with points in specified dataset

In [ ]:
ds = ds_sci
# ds = ds_sci.isel(time=slice(0, None, 10)) # Subsample every 10th point

ds_flagged = ds.sel(time=slice('2026-04-13 01:30:00', '2026-04-13 14:30:00'))

plt.figure(figsize=(9, 6))

# Background track colored by profile_index
sc = plt.scatter(
    ds['longitude'],
    ds['latitude'],
    c=ds['profile_index'],
    cmap='gray',
    s=8,
    alpha=0.5,
    label='Track'
)
plt.colorbar(sc, label='profile_index')

# Overlay flagged points in red on top (higher zorder)
plt.scatter(
    ds_flagged['longitude'],
    ds_flagged['latitude'],
    color='red',
    s=30,
    # edgecolors='black',
    linewidths=0.5,
    label=f'Flagged Points (N={len(ds_flagged["time"]):,})',
    zorder=5
)

plt.xlabel('Longitude (°E)')
plt.ylabel('Latitude (°N)')
plt.title('Glider Track with Flagged Anomaly Points')
plt.legend(loc='best')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

Filter by time slice, and reorder data variables for exploration in Data Explorer extension

In [ ]:
# ds2 = ds_sci.where(ds_sci.time > '2026-04-13 01:00:00', drop=True)
ds2 = ds.sel(time=slice('2026-04-13 13:42:00', '2026-04-13 14:30:00'))

new_start = ["pressure", "profile_index", "latitude", "longitude"]
new_order = new_start + [i for i in ds.data_vars if i not in new_start]
ds2 = ds2[new_order]
# ds2 = utils.data_var_reorder(ds2, new_start)
ds2

#### Plotly

Generate an 'interactive' plot with plotly. 

Items to play with, depoending on what you want to explore:

- The filter. Example filters: depths greater than 5m, up or down profiles
- subsample rate: `time=slice(0, None, 5)`. If you need all points, then you may have to take a time slice of the original dataset (or df)
- color: you can change the variable used to color the data. 
- hover_data: you can add other data to be displayed in the hover_data. To do this, provide '{var_name}': '{formatting}'. For hover data, the most conmmon formatting use case is "f" (Fixed-point): Displays the number as a standard decimal. ":.2f" is a two decimal number, ":.4f" is a four decimal number, etc. 

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Subsample, filter, and extract data (including 'time')
ds_plotly = ds_eng.copy(deep=True)

# filter, if desired
# ds_plotly = ds_plotly.where(ds_plotly.depth < 5, drop=True)
# ds_plotly = ds_plotly.where(ds_plotly.profile_index % 1 == 0, drop=True)

df = ds_plotly.isel(time=slice(0, None, 1))
df = df.to_dataframe().reset_index()

# 2. Format time into a clean string (handles datetime64 or raw epoch seconds)
if pd.api.types.is_datetime64_any_dtype(df['time']):
    df['time_str'] = df['time'].dt.strftime('%Y-%m-%d %H:%M:%S')
else:
    df['time_str'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m-%d %H:%M:%S')

# 3. Render map with time in tooltip
fig = px.scatter_map(
    df,
    lat='latitude',
    lon='longitude',
    color='profile_index',
    color_continuous_scale='Turbo_r',
    size_max=5,
    center={
        'lat': float(df['latitude'].mean()),
        'lon': float(df['longitude'].mean())
    },
    zoom=8,
    map_style='carto-positron',
    hover_name='time_str',  # Shows formatted time as the card title
    hover_data={
        'latitude': ':.4f',
        'longitude': ':.4f',
        'depth': ':.1f',
        'profile_index': ':.1f',
        'time_str': False  # Suppress duplicate entry in the body
    },
    labels={
        'depth': 'Depth (m)',
        'profile_index': 'Profile Index'
    }
)

fig.update_layout(margin={'l': 0, 'r': 0, 'b': 0, 't': 30})
fig.show()